# 02C Orthogonal Signal Factory

Generate a controlled orthogonal signal expansion layer, then promote approved orthogonal signals into the main candidate universe. The sandbox still writes separate `orthogonal_candidate_*` tables first; the final section refreshes only `orthogonal_generated` rows in the main candidate tables and does not write downstream Notebook 03-09 artifacts.

In [1]:
from pathlib import Path
import sqlite3
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "2-Phase 2_Signal Expansion":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.db import load_benchmark_prices, load_ohlcv_panels, load_table, table_exists
from src.orthogonal_signals import (
    ORTHOGONAL_VERSION,
    build_orthogonal_family_summary,
    build_orthogonal_signal_candidates,
    build_orthogonal_signal_quality,
)
from src.orthogonal_signal_storage import (
    MAIN_SIGNAL_TABLES,
    ORTHOGONAL_SIGNAL_TABLES,
    promote_approved_orthogonal_signals_to_main_universe,
    save_orthogonal_signal_outputs,
)
from src.run_config import get_sqlite_db_path, make_run_id, make_run_timestamp

DB_PATH = get_sqlite_db_path()
pd.set_option("display.max_columns", 200)
DB_PATH


PosixPath('/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db')

In [2]:
run_id = make_run_id(prefix="phase2_orthogonal_signals")
created_timestamp = make_run_timestamp()

run_id, created_timestamp, ORTHOGONAL_VERSION

('phase2_orthogonal_signals_20260508_075857',
 '2026-05-08 07:58:57',
 'phase2_orthogonal_signals_v2')

In [3]:
downstream_tables = [
    "signal_scores_current",
    "signal_wfv_candidates_current",
    "signal_decay_summary_current",
    "signal_health_score_current",
    "signal_diversity_selection_current",
    "alpha_construction_metadata_current",
    "constructed_alpha_wfv_gate_current",
    "alpha_stress_gate_current",
    "survivor_alpha_registry_current",
    "portfolio_backtest_summary_current",
]
main_candidate_tables = [
    "candidate_signals_current",
    "candidate_signal_metadata_current",
    "candidate_signal_quality_current",
    "candidate_signal_quality_gate_current",
    "candidate_signal_family_summary_current",
]

def count_table_rows(table_name):
    with sqlite3.connect(DB_PATH) as conn:
        return conn.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]

def count_distinct_signal_names(table_name):
    with sqlite3.connect(DB_PATH) as conn:
        return conn.execute(f'SELECT COUNT(DISTINCT signal_name) FROM "{table_name}"').fetchone()[0]

def load_table_head(table_name, limit=5):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(f'SELECT * FROM "{table_name}" LIMIT {int(limit)}', conn)

def table_row_snapshot(table_names):
    rows = []
    for table_name in table_names:
        exists = table_exists(table_name, db_path=DB_PATH)
        n_rows = count_table_rows(table_name) if exists else pd.NA
        rows.append({"table_name": table_name, "exists": exists, "n_rows": n_rows})
    return pd.DataFrame(rows)

protected_table_snapshot_before = table_row_snapshot(downstream_tables)
main_table_snapshot_before = table_row_snapshot(main_candidate_tables)
main_metadata_before = load_table("candidate_signal_metadata_current", db_path=DB_PATH)
main_signal_names_before = set(main_metadata_before["signal_name"].dropna().astype(str))
n_main_signals_before = int(main_metadata_before["signal_name"].nunique())

print("Downstream protected table snapshot before 02C integration")
display(protected_table_snapshot_before)
print("Main candidate table snapshot before 02C integration")
display(main_table_snapshot_before)


Downstream protected table snapshot before 02C integration


,table_name,exists,n_rows
0,signal_scores_current,True,368
1,signal_wfv_candidates_current,True,15
2,signal_decay_summary_current,True,368
3,signal_health_score_current,True,368
4,signal_diversity_selection_current,True,16
5,alpha_construction_metadata_current,True,10
6,constructed_alpha_wfv_gate_current,True,36
7,alpha_stress_gate_current,True,9
8,survivor_alpha_registry_current,True,7
9,portfolio_backtest_summary_current,False,<NA>


Main candidate table snapshot before 02C integration


,table_name,exists,n_rows
0,candidate_signals_current,True,85241740
1,candidate_signal_metadata_current,True,85
2,candidate_signal_quality_current,True,85
3,candidate_signal_quality_gate_current,True,85
4,candidate_signal_family_summary_current,True,25


In [4]:
ohlcv = load_ohlcv_panels(current=True, db_path=DB_PATH)
benchmark_prices = load_benchmark_prices(current=True, db_path=DB_PATH)

input_shapes = pd.DataFrame(
    [{"input": name, "rows": panel.shape[0], "columns": panel.shape[1]} for name, panel in ohlcv.items()]
    + [{"input": "benchmark_prices", "rows": benchmark_prices.shape[0], "columns": benchmark_prices.shape[1]}]
)
display(input_shapes)

,input,rows,columns
0,open,2098,478
1,high,2098,478
2,low,2098,478
3,close,2098,478
4,volume,2098,478
5,benchmark_prices,2098,1


In [5]:
orthogonal_signals, orthogonal_metadata = build_orthogonal_signal_candidates(
    ohlcv=ohlcv,
    benchmark_prices=benchmark_prices,
    run_id=run_id,
    created_timestamp=created_timestamp,
)
orthogonal_quality = build_orthogonal_signal_quality(
    signals=orthogonal_signals,
    metadata=orthogonal_metadata,
    run_id=run_id,
)
orthogonal_family_summary = build_orthogonal_family_summary(
    quality=orthogonal_quality,
    run_id=run_id,
)

signal_count = len(orthogonal_signals)
cluster_counts = (
    orthogonal_metadata["orthogonal_cluster"]
    .value_counts(dropna=False)
    .rename_axis("orthogonal_cluster")
    .reset_index(name="n_signals")
)
rejected_signals = orthogonal_quality.loc[
    orthogonal_quality["status"].ne("APPROVED_FOR_SCORING")
].copy()
all_nan_signals = orthogonal_quality.loc[orthogonal_quality["is_all_nan"].astype(bool)].copy()

assert 10 <= signal_count <= 20, f"Expected 10-20 orthogonal signals, got {signal_count}."
assert all_nan_signals.empty, "Generated orthogonal signal contains all NaN values."

print(f"Signal count: {signal_count}")
print("Cluster counts")
display(cluster_counts)
print("Metadata sample")
display(orthogonal_metadata.head(10))
print("Quality summary")
display(orthogonal_quality)
print("Rejected / all-NaN signals")
display(rejected_signals)


/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/src/orthogonal_signals.py:244: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns_1d = close.pct_change()
/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/src/orthogonal_signals.py:245: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns_3d = close.pct_change(3)
/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/src/orthogonal_signals.py:246: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to call

Signal count: 15
Cluster counts


,orthogonal_cluster,n_signals
0,cross_sectional_relative_value,3
1,true_short_term_reversal,3
2,volatility_structure,3
3,liquidity_flow,3
4,microstructure_lite,3


Metadata sample


,signal_name,signal_family,orthogonal_cluster,formula_type,required_inputs,lookback,expected_horizon,direction_convention,signal_source,orthogonal_version,run_id,created_timestamp,normalization
0,relative_return_rank_20,cross_sectional_relative_value,cross_sectional_relative_value,cross_sectional_rank_of_20d_return,close,20,5d_to_20d,higher_is_stronger_relative_return,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857,2026-05-08 07:58:57,cross_sectional_rank_centered_by_date
1,relative_return_zscore_60,cross_sectional_relative_value,cross_sectional_relative_value,cross_sectional_zscore_of_60d_return,close,60,20d_to_60d,higher_is_stronger_relative_return,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857,2026-05-08 07:58:57,cross_sectional_zscore_by_date_clipped_3
2,residual_return_vs_universe_20,cross_sectional_relative_value,cross_sectional_relative_value,20d_return_minus_universe_mean_return,close,20,5d_to_20d,higher_is_positive_universe_relative_residual,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857,2026-05-08 07:58:57,cross_sectional_zscore_by_date_clipped_3
3,overnight_gap_reversal_1,true_short_term_reversal,true_short_term_reversal,negative_open_to_prior_close_gap,"open,close",1,1d_to_5d,higher_is_larger_gap_down_reversal_candidate,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857,2026-05-08 07:58:57,cross_sectional_zscore_by_date_clipped_3
4,intraday_reversal_strength_1,true_short_term_reversal,true_short_term_reversal,negative_open_to_close_return,"open,close",1,1d_to_5d,higher_is_larger_intraday_selloff_reversal_can...,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857,2026-05-08 07:58:57,cross_sectional_zscore_by_date_clipped_3
5,three_day_overextension_reversal,true_short_term_reversal,true_short_term_reversal,negative_3d_return_scaled_by_20d_volatility,close,20,1d_to_5d,higher_is_more_overextended_downward,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857,2026-05-08 07:58:57,cross_sectional_zscore_by_date_clipped_3
6,vol_surprise_20_60,volatility_structure,volatility_structure,20d_realized_vol_minus_60d_realized_vol,close,60,5d_to_20d,higher_is_positive_short_vol_surprise,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857,2026-05-08 07:58:57,cross_sectional_zscore_by_date_clipped_3
7,vol_of_vol_20,volatility_structure,volatility_structure,20d_std_of_absolute_daily_returns,close,20,5d_to_20d,higher_is_more_unstable_realized_volatility,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857,2026-05-08 07:58:57,cross_sectional_zscore_by_date_clipped_3
8,range_expansion_failure_5,volatility_structure,volatility_structure,wide_5d_range_with_weak_close_location,"high,low,close",20,1d_to_10d,higher_is_failed_range_expansion,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857,2026-05-08 07:58:57,cross_sectional_zscore_by_date_clipped_3
9,dollar_volume_shock_20,liquidity_flow,liquidity_flow,dollar_volume_vs_trailing_20d_mean,"close,volume",20,1d_to_10d,higher_is_unusual_dollar_volume,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857,2026-05-08 07:58:57,cross_sectional_zscore_by_date_clipped_3


Quality summary


,signal_name,signal_family,orthogonal_cluster,finite_pct,missing_pct,n_dates,n_tickers,first_valid_date,last_valid_date,is_all_nan,is_near_constant,status,quality_notes,orthogonal_version,run_id
0,relative_return_rank_20,cross_sectional_relative_value,cross_sectional_relative_value,0.876450,0.123550,2098,478,2018-03-28,2026-05-07,False,False,APPROVED_FOR_SCORING,Passes basic finite and variation checks.,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857
1,relative_return_zscore_60,cross_sectional_relative_value,cross_sectional_relative_value,0.676087,0.323913,2098,478,2018-06-15,2026-05-07,False,False,APPROVED_FOR_SCORING,Passes basic finite and variation checks.,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857
2,residual_return_vs_universe_20,cross_sectional_relative_value,cross_sectional_relative_value,0.638608,0.361392,2098,478,2018-04-19,2026-05-07,False,False,APPROVED_FOR_SCORING,Passes basic finite and variation checks.,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857
3,overnight_gap_reversal_1,true_short_term_reversal,true_short_term_reversal,0.889357,0.110643,2098,478,2018-02-23,2026-05-07,False,False,APPROVED_FOR_SCORING,Passes basic finite and variation checks.,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857
4,intraday_reversal_strength_1,true_short_term_reversal,true_short_term_reversal,0.892692,0.107308,2098,478,2018-02-22,2026-05-07,False,False,APPROVED_FOR_SCORING,Passes basic finite and variation checks.,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857
5,three_day_overextension_reversal,true_short_term_reversal,true_short_term_reversal,0.888157,0.111843,2098,478,2018-03-08,2026-05-07,False,False,APPROVED_FOR_SCORING,Passes basic finite and variation checks.,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857
6,vol_surprise_20_60,volatility_structure,volatility_structure,0.871368,0.128632,2098,478,2018-05-04,2026-05-07,False,False,APPROVED_FOR_SCORING,Passes basic finite and variation checks.,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857
7,vol_of_vol_20,volatility_structure,volatility_structure,0.880582,0.119418,2098,478,2018-04-06,2026-05-07,False,False,APPROVED_FOR_SCORING,Passes basic finite and variation checks.,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857
8,range_expansion_failure_5,volatility_structure,volatility_structure,0.872500,0.127500,2098,478,2018-03-07,2026-05-07,False,False,APPROVED_FOR_SCORING,Passes basic finite and variation checks.,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857
9,dollar_volume_shock_20,liquidity_flow,liquidity_flow,0.826633,0.173367,2098,478,2018-05-04,2026-05-07,False,False,APPROVED_FOR_SCORING,Passes basic finite and variation checks.,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_20260508_075857


Rejected / all-NaN signals


,signal_name,signal_family,orthogonal_cluster,finite_pct,missing_pct,n_dates,n_tickers,first_valid_date,last_valid_date,is_all_nan,is_near_constant,status,quality_notes,orthogonal_version,run_id


In [6]:
saved_paths = save_orthogonal_signal_outputs(
    signals=orthogonal_signals,
    metadata=orthogonal_metadata,
    quality=orthogonal_quality,
    family_summary=orthogonal_family_summary,
    db_path=DB_PATH,
    run_id=run_id,
    orthogonal_version=ORTHOGONAL_VERSION,
)

sandbox_artifacts = ["signals", "metadata", "quality", "family_summary"]
orthogonal_sqlite_tables_written = pd.DataFrame(
    [
        {
            "artifact": artifact,
            "current_table": ORTHOGONAL_SIGNAL_TABLES[artifact][0],
            "history_table": ORTHOGONAL_SIGNAL_TABLES[artifact][1],
            "sqlite_path": str(saved_paths[artifact]),
        }
        for artifact in sandbox_artifacts
    ]
)

display(orthogonal_sqlite_tables_written)


,artifact,current_table,history_table,sqlite_path
0,signals,orthogonal_candidate_signals_current,orthogonal_candidate_signals_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,metadata,orthogonal_candidate_metadata_current,orthogonal_candidate_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,quality,orthogonal_candidate_quality_current,orthogonal_candidate_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,family_summary,orthogonal_candidate_family_summary_current,orthogonal_candidate_family_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## Promote Approved Orthogonal Signals to Main Candidate Universe

Append approved orthogonal signals into the main candidate signal universe with signal-name and signal/date/ticker dedupe. Downstream Notebook 03-09 tables are checked before and after but are not written here.

In [7]:
approved_orthogonal_quality = load_table("orthogonal_candidate_quality_current", db_path=DB_PATH).loc[
    lambda df: df["status"].eq("APPROVED_FOR_SCORING")
].copy()
approved_orthogonal_signal_names = sorted(approved_orthogonal_quality["signal_name"].dropna().astype(str).unique())

integration_report, integration_diagnostics = promote_approved_orthogonal_signals_to_main_universe(
    db_path=DB_PATH,
    run_id=run_id,
    orthogonal_version=ORTHOGONAL_VERSION,
)

main_metadata_after = load_table("candidate_signal_metadata_current", db_path=DB_PATH)
main_quality_after = load_table("candidate_signal_quality_current", db_path=DB_PATH)
main_family_summary_after = load_table("candidate_signal_family_summary_current", db_path=DB_PATH)
main_table_snapshot_after = table_row_snapshot(main_candidate_tables)
protected_table_snapshot_after = table_row_snapshot(downstream_tables)

main_signal_names_after = set(main_metadata_after["signal_name"].dropna().astype(str))
orthogonal_present_in_metadata = set(approved_orthogonal_signal_names).issubset(main_signal_names_after)
existing_main_signals_still_present = main_signal_names_before.issubset(main_signal_names_after)
n_main_signals_after = int(main_metadata_after["signal_name"].nunique())
n_orthogonal_added = int(integration_report["n_orthogonal_added"].iloc[0])

protected_table_check = protected_table_snapshot_before.merge(
    protected_table_snapshot_after,
    on="table_name",
    suffixes=("_before", "_after"),
)
protected_table_check["unchanged"] = (
    protected_table_check["exists_before"].fillna(False).eq(protected_table_check["exists_after"].fillna(False))
    & protected_table_check["n_rows_before"].fillna(-1).eq(protected_table_check["n_rows_after"].fillna(-1))
)
downstream_tables_unchanged_check = bool(protected_table_check["unchanged"].all())

with sqlite3.connect(DB_PATH) as conn:
    duplicate_metadata_names = conn.execute(
        """
        SELECT COUNT(*)
        FROM (
            SELECT signal_name, COUNT(*) AS n_rows
            FROM candidate_signal_metadata_current
            GROUP BY signal_name
            HAVING n_rows > 1
        )
        """
    ).fetchone()[0]
    candidate_signal_name_query = pd.read_sql_query(
        "SELECT DISTINCT signal_name FROM candidate_signals_current WHERE signal_name IN (" + ",".join("?" for _ in approved_orthogonal_signal_names) + ")",
        conn,
        params=approved_orthogonal_signal_names,
    ) if approved_orthogonal_signal_names else pd.DataFrame(columns=["signal_name"])
    duplicate_approved_signal_rows = conn.execute(
        f"""
        SELECT COUNT(*)
        FROM (
            SELECT signal_name, Date, ticker, COUNT(*) AS n_rows
            FROM candidate_signals_current
            WHERE signal_name IN ({','.join('?' for _ in approved_orthogonal_signal_names)})
            GROUP BY signal_name, Date, ticker
            HAVING n_rows > 1
        )
        """,
        approved_orthogonal_signal_names,
    ).fetchone()[0] if approved_orthogonal_signal_names else 0

main_signal_source_counts = (
    main_metadata_after.get("signal_source", pd.Series("", index=main_metadata_after.index))
    .fillna("unknown")
    .replace("", "unknown")
    .value_counts(dropna=False)
    .rename_axis("signal_source")
    .reset_index(name="n_signals")
)
main_family_counts = (
    main_metadata_after["signal_family"]
    .value_counts(dropna=False)
    .rename_axis("signal_family")
    .reset_index(name="n_signals")
)
quality_counts = (
    load_table("orthogonal_candidate_quality_current", db_path=DB_PATH)["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n_signals")
)

integration_sqlite_tables_written = pd.DataFrame(
    [
        {
            "artifact": "integration_report",
            "current_table": ORTHOGONAL_SIGNAL_TABLES["integration_report"][0],
            "history_table": ORTHOGONAL_SIGNAL_TABLES["integration_report"][1],
            "sqlite_path": str(DB_PATH),
        },
        *[
            {
                "artifact": artifact,
                "current_table": tables[0],
                "history_table": tables[1],
                "sqlite_path": str(DB_PATH),
            }
            for artifact, tables in MAIN_SIGNAL_TABLES.items()
        ],
    ]
)

verification_summary = pd.DataFrame(
    [
        {"check": "02c_reruns_end_to_end", "passed": True, "details": "Notebook execution reached integration verification."},
        {"check": "existing_main_signals_remain_present", "passed": existing_main_signals_still_present, "details": f"before={len(main_signal_names_before)}, after={len(main_signal_names_after)}"},
        {"check": "approved_orthogonal_present_in_candidate_signals", "passed": set(approved_orthogonal_signal_names).issubset(set(candidate_signal_name_query["signal_name"].dropna().astype(str))), "details": f"approved={len(approved_orthogonal_signal_names)}"},
        {"check": "approved_orthogonal_present_in_metadata", "passed": orthogonal_present_in_metadata, "details": f"approved={len(approved_orthogonal_signal_names)}"},
        {"check": "approved_orthogonal_present_in_quality", "passed": set(approved_orthogonal_signal_names).issubset(set(main_quality_after["signal_name"].dropna().astype(str))), "details": f"approved={len(approved_orthogonal_signal_names)}"},
        {"check": "main_signal_count_increased_by_added_non_duplicates", "passed": n_main_signals_after == n_main_signals_before + n_orthogonal_added, "details": f"before={n_main_signals_before}, added={n_orthogonal_added}, after={n_main_signals_after}"},
        {"check": "no_duplicate_signal_names_in_metadata", "passed": duplicate_metadata_names == 0, "details": f"duplicates={duplicate_metadata_names}"},
        {"check": "no_duplicate_approved_signal_date_ticker_rows", "passed": duplicate_approved_signal_rows == 0, "details": f"duplicates={duplicate_approved_signal_rows}"},
        {"check": "downstream_tables_unchanged_check", "passed": downstream_tables_unchanged_check, "details": "Tracked downstream table row counts unchanged."},
    ]
)

print("Orthogonal quality counts")
display(quality_counts)
print("Approved orthogonal signal names")
display(pd.DataFrame({"signal_name": approved_orthogonal_signal_names}))
print("Integration report")
display(integration_report)
print("Updated main signal source counts")
display(main_signal_source_counts)
print("Updated main family counts")
display(main_family_counts)
print("Verification checks")
display(verification_summary)
print("Downstream tables unchanged check")
display(protected_table_check)
print("SQLite tables written")
display(pd.concat([orthogonal_sqlite_tables_written, integration_sqlite_tables_written], ignore_index=True))


/var/folders/4p/d0pjwrwn08n_6l2vtvsdnzl00000gn/T/ipykernel_93890/3711903105.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & protected_table_check["n_rows_before"].fillna(-1).eq(protected_table_check["n_rows_after"].fillna(-1))


Orthogonal quality counts


,status,n_signals
0,APPROVED_FOR_SCORING,15


Approved orthogonal signal names


,signal_name
0,close_position_reversal_5
1,dollar_volume_shock_20
2,failed_breakout_reversal_20
3,intraday_reversal_strength_1
4,liquidity_adjusted_reversal_5
5,overnight_gap_reversal_1
6,price_impact_proxy_20
7,range_compression_breakout_10
8,range_expansion_failure_5
9,relative_return_rank_20


...

,check,passed,details
0,02c_reruns_end_to_end,True,Notebook execution reached integration verific...
1,existing_main_signals_remain_present,True,"before=85, after=100"
2,approved_orthogonal_present_in_candidate_signals,True,approved=15
3,approved_orthogonal_present_in_metadata,True,approved=15
4,approved_orthogonal_present_in_quality,True,approved=15
5,main_signal_count_increased_by_added_non_dupli...,True,"before=85, added=15, after=100"
6,no_duplicate_signal_names_in_metadata,True,duplicates=0
7,no_duplicate_approved_signal_date_ticker_rows,True,duplicates=0
8,downstream_tables_unchanged_check,True,Tracked downstream table row counts unchanged.


Downstream tables unchanged check


,table_name,exists_before,n_rows_before,exists_after,n_rows_after,unchanged
0,signal_scores_current,True,368,True,368,True
1,signal_wfv_candidates_current,True,15,True,15,True
2,signal_decay_summary_current,True,368,True,368,True
3,signal_health_score_current,True,368,True,368,True
4,signal_diversity_selection_current,True,16,True,16,True
5,alpha_construction_metadata_current,True,10,True,10,True
6,constructed_alpha_wfv_gate_current,True,36,True,36,True
7,alpha_stress_gate_current,True,9,True,9,True
8,survivor_alpha_registry_current,True,7,True,7,True
9,portfolio_backtest_summary_current,False,<NA>,False,<NA>,True


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,signals,orthogonal_candidate_signals_current,orthogonal_candidate_signals_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,metadata,orthogonal_candidate_metadata_current,orthogonal_candidate_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,quality,orthogonal_candidate_quality_current,orthogonal_candidate_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,family_summary,orthogonal_candidate_family_summary_current,orthogonal_candidate_family_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,integration_report,orthogonal_signal_integration_report_current,orthogonal_signal_integration_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,signals,candidate_signals_current,candidate_signals_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
6,metadata,candidate_signal_metadata_current,candidate_signal_metadata_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
7,quality,candidate_signal_quality_current,candidate_signal_quality_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
8,quality_gate,candidate_signal_quality_gate_current,candidate_signal_quality_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
9,family_summary,candidate_signal_family_summary_current,candidate_signal_family_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
